In [2]:

import gradio as gr
import time
from datetime import datetime
custom_css = """
# /////////////////////////////////////////////////////////////////////////////////////////
/* Sidebar and layout styling */

#fixed-search-bar {
    position: sticky;
    top: 0;
    z-index: 1000;
    background: white;
    padding: 10px 0;
    border-bottom: 1px solid #e2e8f0;
}

/* REQUIRED FOR STICKY TO WORK */
.gradio-container {
    height: 100vh;
    overflow-y: auto;
}


#sidebar-toggle {
    display: flex;
    align-items: center;
    justify-content: center;
    width: 40px;
    height: 40px;
    border-radius: 8px;
    background: #f1f5f9;
    border: 1px solid #e2e8f0;
    cursor: pointer;
    transition: all 0.3s ease;
}

#sidebar-toggle:hover {
    background: #e2e8f0;
    transform: scale(1.05);
}

.quick-action-btn {
    width: 100%;
    margin: 5px 0;
    text-align: left;
    padding: 8px 12px;
}

.hamburger-icon {
    font-size: 20px;
    font-weight: bold;
}

/* Scrollable chatbot area */
#chatbot-container {
    height: 600px;
    overflow-y: auto !important;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    padding: 15px;
    background: #f8fafc;
}

/* Custom scrollbar for chatbot */
#chatbot-container::-webkit-scrollbar {
    width: 8px;
}

#chatbot-container::-webkit-scrollbar-track {
    background: #f1f5f9;
    border-radius: 4px;
}

#chatbot-container::-webkit-scrollbar-thumb {
    background: #cbd5e1;
    border-radius: 4px;
}

#chatbot-container::-webkit-scrollbar-thumb:hover {
    background: #94a3b8;
}

/* Chatbot styling */
.chatbot {
    min-height: 600px !important;
    overflow-y: auto !important;
}

/* Sidebar animation */
.sidebar-content {
    transition: all 0.3s ease;
    overflow: hidden;
}

/* Responsive design */
@media (max-width: 768px) {
    .chat-container {
        padding: 0 5px;
    }

    #chatbot-container {
        height: 500px;
    }
}
/* Fix for Dataframe height */
#data-preview-table {
    height: 300px !important;
    overflow-y: auto !important;
}

/* Make dataframe scrollable */
.gr-dataframe {
    max-height: 300px !important;
    overflow-y: auto !important;
}

/* Fix for any other dataframes */
div[class*="dataframe"] {
    max-height: 300px !important;
    overflow-y: auto !important;
}
"""



from transformers import TextIteratorStreamer
from threading import Thread
import torch

domain_cache = {}   # {"Admissions": {"chunks":..., "index":...}}

def load_domain(domain):
    file_path = f"{domain}.pkl"
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    return data["chunks"], data["index"]


def retrieve(query, domain, top_k=3):
    # 1️⃣ Load domain if not cached
    if domain not in domain_cache:
        chunks, index = load_domain(domain)
        domain_cache[domain] = {
            "chunks": chunks,
            "index": index
        }
    else:
        chunks = domain_cache[domain]["chunks"]
        index = domain_cache[domain]["index"]

    # 2️⃣ Embed query
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    # 3️⃣ Search only inside selected domain index
    scores, indices = index.search(query_embedding, top_k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    scores = scores[0].tolist()

    return retrieved_chunks, scores

def generate_answer_stream(query, history, domain="Auto-Detect", search_mode="Shallow"):
    if search_mode == "Shallow":
        k = 3
        min_score = 0.1
        temperature = 0.5
        top_p = 0.9
        max_tokens = 101
        min_tokens = 10
    else:  # Deep
        k = 10
        min_score = 0.05
        # temperature = 0.85
        # top_p = 0.95
        temperature = 0.5
        top_p = 0.9
        max_tokens = 300
        min_tokens = 50


    # 1. Retrieve
    # retrieved_chunks, scores = retrieve(query, k)
    # ////////////////////////////////////////////////////////////////////
    retrieved_chunks, scores = retrieve(query, domain=domain, top_k=k)

    # 2. Score filtering
    filtered_chunks = [
        chunk for chunk, score in zip(retrieved_chunks, scores)
        if score >= min_score
    ]

    # 3. Force-include critical info
    # for chunk in chunks:
    #     if ("doctoral" in chunk.lower() or "deng" in chunk.lower()) \
    #        and chunk not in filtered_chunks:
    #         filtered_chunks.append(chunk)
    for chunk in retrieved_chunks:
     if ("doctoral" in chunk.lower() or "deng" in chunk.lower()) \
       and chunk not in filtered_chunks:
        filtered_chunks.append(chunk)

    # 4. Context fallback
    # context = "\n".join(filtered_chunks) if filtered_chunks else "\n".join(retrieved_chunks)
    context_text = "\n".join(filtered_chunks) if filtered_chunks else "\n".join(retrieved_chunks)
    # Build conversation history string

    MAX_HISTORY = 10  # number of user+assistant turns to keep
    truncated_history = history[-MAX_HISTORY*2:]  # each turn has user+assistant

    conversation_context = ""
    # for turn in history:
    for turn in truncated_history:
      role = turn["role"]
      content = turn["content"]
      if role == "user":
        conversation_context += f"<|user|>\n{content}\n<|end|>\n"
      else:
        conversation_context += f"<|assistant|>\n{content}\n<|end|>\n"

    # 5. Prompt
    if search_mode == "Shallow":
        context_instruction = """
Use the provided context to answer the user's question accurately.
Answer shortly and completely. Keep within 30 tokens.
"""
    else:
        context_instruction = """
Use the provided context to answer the user's question accurately.
Answer in detail, thoroughly covering all relevant points.
"""

    prompt = f"""
<|system|>
You are a helpful assistant. Use the provided context to answer the user's question accurately.
You are an AI assistant created by M Sajjad.
When anyone asks who made you, always reply: "I was created by M Sajjad."
{context_instruction}
<|end|>



Context:
{context_text}

Conversation history:
{conversation_context}

Current Question:
{query}
<|end|>



<|assistant|>
"""

    # 6. Tokenize
    inputs = hf_tokenizer(prompt, return_tensors="pt").to(model.device)

    # 7. Streaming
    streamer = TextIteratorStreamer(hf_tokenizer, skip_special_tokens=True)

    # 8. Generate (async)
    Thread(target=model.generate, kwargs={
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_tokens,
        "min_new_tokens": min_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "do_sample": True,
        "use_cache": True
    }).start()


    # 9. Yield tokens
    partial_text = ""
    for new_text in streamer:
        if "<|assistant|>" in new_text:
            new_text = new_text.split("<|assistant|>")[-1]
        if "<|end|>" in new_text:
            new_text = new_text.split("<|end|>")[0]

        partial_text += new_text
        yield partial_text


# ========================================
# Chat handler (GUI / API CONNECTOR)
# ========================================
def chat_with_bot(message, history, domain="Auto-Detect", mode="Shallow"):
    history = history or []

    # Show only user message in UI
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": ""})

    # Prepare query for backend (internal)
    # query_for_model = f"Domain: {domain} | Mode: {mode}\n{message}"

    # for partial_answer in generate_answer_stream(query_for_model,history,domain=domain,search_mode=mode):
    for partial_answer in generate_answer_stream(message, history, domain=domain, search_mode=mode):
        history[-1]["content"] = partial_answer
        yield history, history, ""






def toggle_sidebar(visible):
    """Toggle sidebar visibility"""
    return not visible, gr.update(visible=not visible)


In [3]:
def build_chat_tab(tab_id, domain_choices, quick_action_texts, sidebar_visible):
    with gr.TabItem(tab_id):
        # ================= SEARCH BAR =================
        with gr.Row(elem_id="fixed-search-bar"):
            with gr.Column(scale=4):
                chat_search = gr.Textbox(
                    placeholder="Type your question here...",
                    show_label=False,
                    container=False
                )
        # ================= SEARCH BAR =================

            with gr.Column(scale=1):
                search_mode = gr.Radio(
                    choices=["Shallow", "Deep"],
                    value="Shallow",
                    label="Search Mode",
                    container=False
                )
            with gr.Column(scale=1):
                chat_send_btn = gr.Button("Search 🔍", variant="primary")

        # ================= MAIN AREA =================
        with gr.Row():
            with gr.Column(scale=1, min_width=250, visible=False) as sidebar:
                gr.Markdown("### Configuration")

                domain_dropdown = gr.Dropdown(
                    choices=domain_choices,
                    value="Auto-Detect",
                    label="Select Domain"
                )

                gr.Markdown("---")
                gr.Markdown("### Quick Actions")

                quick_buttons = []
                for text in quick_action_texts:
                    btn = gr.Button(
                        text,
                        elem_classes="quick-action-btn",
                        variant="secondary"                    )
                    quick_buttons.append(btn)

            # Chat area
            with gr.Column(scale=4, elem_classes="chat-container"):
                with gr.Row():
                    toggle_btn = gr.Button(
                        value="☰",
                        size="sm"
                    )

                chatbot = gr.Chatbot(
                    type="messages",
                    height=600,
                    show_label=False
                )
                chat_history = gr.State([])

        # ================= EVENTS =================
        toggle_btn.click(
            toggle_sidebar,
            inputs=[sidebar_visible],
            outputs=[sidebar_visible, sidebar]
        )

        chat_send_btn.click(
            chat_with_bot,
            inputs=[chat_search, chat_history, domain_dropdown, search_mode],
            # inputs=[chat_search, chat_history],
            outputs=[chatbot, chat_history, chat_search]
        )

        chat_search.submit(
            chat_with_bot,
            inputs=[chat_search, chat_history, domain_dropdown, search_mode],
            # inputs=[chat_search, chat_history],
            outputs=[chatbot, chat_history, chat_search]
        )

        # Fix late binding lambda in loop
        for btn in quick_buttons:
            btn.click(
                fn=lambda x, b=btn: b.value,
                inputs=[btn],
                outputs=[chat_search]
            )

## Admin Panel code and its working logic

In [ ]:
import gradio as gr
import os
import time
import random

def build_admin_left_panel_ta22bb():
    with gr.TabItem("⚙️ Admin Panel", id="admin_content"):
        gr.Markdown("### System Administration and Configuration")

        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                gr.Markdown("**Admin Controls**")

                # Step 1: Update Database button
                update_btn = gr.Button("🔄 Update Database", variant="primary")

                # State for preview toggle
                preview_visible_state = gr.State(False)

                with gr.Column(visible=False) as update_db_panel:
                    gr.Markdown("### Update Database Panel")

                    # Step 2: Upload Document button
                    upload_doc_btn = gr.Button("📤 Upload Document")

                    # Step 3: Document type dropdown
                    doc_type_dropdown = gr.Dropdown(
                        choices=["Select Docs Type", "Chat Assistant", "Events", "General"],
                        label="Select Document Type",
                        visible=False
                    )

                    # Step 4: Title input (manual typing or dropdown)
                    title_input = gr.Dropdown(
                        choices=["Admissions", "Recruitment", "Staff", "Schedules", "Events"],
                        label="Title (type or select)",
                        interactive=True,
                        allow_custom_value=True,
                        visible=False
                    )

                    # Step 5: Event date (only for Events)
                    event_date_input = gr.Textbox(
                        placeholder="Enter Event Date (YYYY-MM-DD)",
                        label="Event Date",
                        visible=False
                    )

                    # Step 6: Select document from system
                    select_doc_btn = gr.File(
                        label="📂 Select Document from PS",
                        file_types=[".txt"],
                        type="filepath",
                        visible=False
                    )

                    # Preview button (shows/hides file content)
                    preview_doc_btn = gr.Button(
                        "👁️ Document Content Preview",


                # Toggle preview function
                def toggle_preview(file_path, visible):
                    if not file_path:
                        return gr.update(value="No file selected.", visible=True), not visible
                    if visible:
                        return gr.update(value="", visible=False), False
                    else:
                        try:
                            with open(file_path, "r", encoding="utf-8") as f:
                                content = f.read()
                        except Exception as e:
                            content = f"Error reading file: {e}"
                        return gr.update(value=content, visible=True), True

                preview_doc_btn.click(
                    toggle_preview,
                    inputs=[select_doc_btn, preview_visible_state],
                    outputs=[document_preview, preview_visible_state]
                )

                # Function to show status panel when trigger is clicked
                def show_status_panel():
                    return gr.update(visible=True)

                trigger_prepare_btn.click(
                    show_status_panel,
                    inputs=[],
                    outputs=[status_panel]
                )

                # ==================== AUTO-TRIGGER FUNCTION ====================
                # This is a generator function for real-time status/progress updates
                def auto_trigger_pipeline(file_path, domain_name, trigger_btn):
                    # Show initial status and disable ONLY trigger button
                    yield (
                        "🚀 Starting pipeline...",  # Direct string for status_box
                        0,                          # Direct number for progress_bar
                        gr.update(inte)

                    # STEP 1: Cleaning (20%)
                    for i in range(1, 21):
                        status = f"1️⃣ Cleaning Text\n🔧 Status: Cleaning and normalizing text for document: {document_name}\n📊 Progress: {i}%"
                        yield status, i, gr.update(interactive=False), gr.update(interactive=False)
                        time.sleep(random.uniform(0.05, 0.15))  # Random gradual increment

                    completed_steps += 1
                    time.sleep(random.uniform(0.5, 1.0))  # Random processing time

                    # STEP 2: Chunking + Embedding (20% → 40%)
                    for i in range(21, 41):
                        status = f"2️⃣ Processing Document\n📄 Processing document: {document_name}\n📊 Progress: {i}%"
                        yield status, i, gr.update(interactive=False), gr.update(interactive=False)
                        time.sleep(random.uniform(0.05, 0.15))

                    # Call the actual pipeline function
                    cleaned_docs, chunks, embeddings = full_pipeline(document_name, domain_name)
                    completed_steps += 1
                    time.sleep(random.uniform(1.0, 2.0))  # Random processing time

                    # STEP 3: Generating Embeddings (40% → 60%)
                    for i in range(41, 61):
                        status = f"3️⃣ Generating Embeddings\n⚡ Generating embeddings for document: {document_name}\n📊 Progress: {i}%"
                        yield status, i, gr.update(interactive=False), gr.update(interactive=False)
                        time.sleep(random.uniform(0.05, 0.15))

                    completed_steps += 1
                    time.sleep(random.uniform(0.8, 1.5))  # Random processing time

                    # STEP 4: Storing in ChromaDB (60% → 80%)
                    for i in range(61, 81):
                        status = f"4️⃣ Storing in Database\n💾 Storing chunks and embeddings in ChromaDB collection: {domain_name}\n📊 Progress: {i}%"
                        yield status, i, gr.update(interactive=False), gr.update(interactive=False)
                        time.sleep(random.uniform(0.05, 0.15))

                    completed_steps += 1
                    time.sleep(random.uniform(1.0, 2.0))  # Random processing time

                    # STEP 5: Completed (80% → 100%)
                    for i in range(81, 101):
                        status = f"5️⃣ Document Completed\n✅ Document {document_name} successfully added to {domain_name} collection\n📊 Progress: {i}%"
                        yield status, i, gr.update(interactive=False), gr.update(interactive=True)  # Enable Save
                        time.sleep(random.uniform(0.05, 0.15))

                    completed_steps += 1

                    # Final message
                    final_status = f"""✅ PROCESSING COMPLETE 🎉

📊 Summary:
• 📄 Document: {document_name}
• 🏷️ Domain: {domain_name}
• ✅ Status: Successfully processed and stored
• 🤖 Chatbot is ready for queries!

📈 Upload complete. Chatbot ready for queries.

💡 Click '💾 Save Document' to finalize."""
                    yield final_status, 100, gr.update(interactive=False), gr.update(interactive=True)

                # Bind Auto-Trigger button with ONLY 4 outputs (removed extra button controls)
                trigger_prepare_btn.click(
                    lambda: gr.update(interactive=False),  # Initially disable save button
                    inputs=[],
                    outputs=[save_doc_btn]
                ).then(
                    auto_trigger_pipeline,
                    inputs=[select_doc_btn, title_input, trigger_prepare_btn],
                    outputs=[
                        status_box,
                        progress_bar,
                        trigger_prepare_btn,
                        save_doc_btn  # Only these 4 outputs
                    ]
                )

                # Save Document button click - re-enable ONLY the Trigger button
                def on_save_document_click():
                    """
                    Handle Save Document button click - only re-enable Trigger button
                    """
                    return (
                        gr.update(interactive=True),   # trigger_prepare_btn - RE-ENABLE
                        gr.update(interactive=True)    # save_doc_btn - keep enabled
                    )

                save_doc_btn.click(
                    on_save_document_click,
                    inputs=[],
                )

                #trigger_btn.gradio-button:disabled {
                    background: linear-gradient(135deg, #cccccc 0%, #999999 100%) !important;
                    box-shadow: none !important;
                    cursor: not-allowed !important;
                }

                #save_btn.gradio-button {
                    background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%) !important;
                    border: none !important;
                    color: white !important;
                    font-weight: 600 !important;
                    padding: 12px 24px !important;
                    border-radius: 25px !important;
                    box-shadow: 0 4px 15px rgba(67, 233, 123, 0.4) !important;
                    transition: all 0.3s ease !important;
                }

                #save_btn.gradio-button:hover {
                    transform: translateY(-2px) !important;
                    box-shadow: 0 6px 20px rgba(67, 233, 123, 0.6) !important;
                }

                #save_btn.gradio-button:disabled {
                    background: linear-gradient(135deg, #cccccc 0%, #999999 100%) !important;
                    box-shadow: none !important;
                    cursor: not-allowed !important;
                }

                /* Panel Styling */
                .gradio-tabitem {
                    border-radius: 15px !important;
                }

                .gradio-column {
                    border-radius: 12px !important;
                    background: linear-gradient(135deg, #ffffff 0%, #f8f9fa 100%) !important;
                    padding: 20px !important;
                    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.05) !important;
                }

                /* Status Panel */
                .gradio-column:has(#modern_progress_bar) {
                    background: linear-gradient(135deg, #ffffff 0%, #f0f4f8 100%) !important;
                    border: 2px solid #e3f2fd !important;
                }

                /* Smooth transitions */
                .gradio-button, .gradio-dropdown, .gradio-textbox, .gradio-file {
                    transition: all 0.3s ease !important;
                }
                """

                # Add CSS to the interface
                gr.HTML(f"<style>{css}</style>")

### Cleaning text

In [13]:
def clean_text_step(user_input):
    import re
    from typing import Dict

    # -----------------------------
    # Cleaning a single text
    # -----------------------------
    def clean_text(text: str) -> str:
        if not text:
            return ""
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[^A-Za-z0-9.,;:!?\'"()\- ]+', '', text)
        text = text.replace("“", '"').replace("”", '"').replace("–", "-")
        text = text.lower()
        text = text.encode('utf-8', 'ignore').decode('utf-8')
        sentences = re.split(r'(?<=[.!?])\s+', text)
        cleaned_sentences = [s for s in sentences if not re.match(r'^(page|confidential)', s.strip())]
        return " ".join(s.strip() for s in cleaned_sentences)

    # -----------------------------
    # Clean multiple documents
    # -----------------------------
    def clean_documents(documents: Dict[str, str], verbose: bool = True) -> Dict[str, str]:
        total_docs = len(documents)
        cleaned_docs = {}
        for i, (doc_name, doc_text) in enumerate(documents.items(), start=1):
            if verbose:
                # print(f"[Step 1: Cleaning] Processing document {i}/{total_docs}: '{doc_name}'")
                print(" ")
            cleaned_text = clean_text(doc_text)
            cleaned_docs[doc_name] = cleaned_text
            if verbose:
                # print(f"[Step 1: Cleaning] Document '{doc_name}' cleaned. Length: {len(cleaned_text)} characters")
                print("")
        if verbose:
            # print(f"[Step 1: Cleaning] All {total_docs} documents cleaned successfully.")
            print("")
        return cleaned_docs

    # -----------------------------
    # Read a text file
    # -----------------------------

    def read_text_file(path):
      with open(path, "r", encoding="utf-8") as f:
            return f.read()
    file_name = user_input
    if not file_name.lower().endswith(".txt"):
      file_name = f"{file_name}.txt"
    documents = {file_name: read_text_file(file_name)}






# Use a new variable instead of modifying user_input




# Load document



    # Step 2: Clean documents
    cleaned_documents = clean_documents(documents)

    # Step 3: Display cleaned text
    # for name, text in cleaned_documents.items():
    #     print(f"\n{name}:\n{text}")

    # Step 4: Return cleaned documents
    return cleaned_documents

### Chunking

In [14]:
import tiktoken
from typing import List, Dict

# =========================
# Tokenizer Initialization
# =========================
tokenizer = tiktoken.get_encoding("cl100k_base")

# =========================
# Chunk Size Logic
# =========================
def get_chunk_params(total_tokens: int):
    if total_tokens < 300:
        chunk_size = total_tokens
        overlap = 0
    elif 300 <= total_tokens <= 500:
        chunk_size = 250
        overlap = 40
    elif 500 < total_tokens <= 800:
        chunk_size = 350
        overlap = 60
    elif 800 < total_tokens <= 1200:
        chunk_size = 400
        overlap = 70
    elif 1200 < total_tokens <= 1500:
        chunk_size = 450
        overlap = 90
    else:
        chunk_size = 500
        overlap = 100

    return chunk_size, overlap

# =========================
# Chunking Function as a single callable function
# =========================
def chunking_step(cleaned_docs: Dict[str, str], domain_name: str) -> List[Dict]:
    """
    Chunk cleaned documents into smaller segments using tiktoken.

    Args:
        cleaned_docs (dict): dictionary of {filename: cleaned_text}
        domain_name (str): domain name to assign to chunks

    Returns:
        List[Dict]: list of chunks with domain, chunk_id, and text
    """
    all_chunks = []

    for filename, doc_text in cleaned_docs.items():
        # 1️⃣ Tokenize document
        tokens = tokenizer.encode(doc_text)
        total_tokens = len(tokens)
        # ...............................................
        # print(f"Total tokens in '{filename}': {total_tokens}")
        # print(f"[Step 2: Chunking] Total tokens: {total_tokens}")

        # 2️⃣ Get chunk parameters
        chunk_size, overlap = get_chunk_params(total_tokens)
        step_size = chunk_size - overlap
        # print(f"[Step 2: Chunking] Chunk size: {chunk_size}, Overlap: {overlap}")

        # 3️⃣ Create chunks
        chunks = []
        chunk_id = 0
        for start in range(0, total_tokens, step_size):
            end = start + chunk_size
            chunk_tokens = tokens[start:end]
            if not chunk_tokens:
                break
            chunk_text = tokenizer.decode(chunk_tokens)

            chunks.append({
                "domain": domain_name,
                "chunk_id": chunk_id,
                "text": chunk_text
            })
            chunk_id += 1

        # print(f"[Step 2: Chunking] Total chunks created for '{filename}': {len(chunks)}")
        all_chunks.extend(chunks)

    # =========================
    # Display Chunks
    # =========================
    # for chunk in all_chunks:
    #     # print("\n---------------------")
    #     # print(f"Chunk {chunk['chunk_id']}:")
    #     # print(f"Domain: {chunk['domain']}")
    #     # print(f"Text: {chunk['text']}")

    return all_chunks


### Embeding

In [15]:
# !pip install -U sentence-transformers torch
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict, Tuple

def embedding_step(chunks: List[Dict]) -> Tuple[np.ndarray, List[str], List[Dict], List[str]]:
    """
    Generate embeddings for document chunks using SentenceTransformer.

    Args:
        chunks (List[Dict]): List of chunk dictionaries with 'text', 'chunk_id', 'domain'

    Returns:
        Tuple[np.ndarray, List[str], List[Dict], List[str]]: embeddings, texts, metadatas, ids
    """

    # Initialize the embedding model
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    # Extract texts from chunks
    texts = [chunk["text"] for chunk in chunks]

    # Prepare metadata and IDs
    metadatas = [
        {
            # "domain": chunk["domain"],  # original code commented this out
            "chunk_id": chunk["chunk_id"]
        }
        for chunk in chunks
    ]

    ids = [f"{chunk['domain']}_{chunk['chunk_id']}" for chunk in chunks]

    # Generate embeddings
    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True
        # ).tolist()  # original code had this commented
    )
    embeddings = np.array(embeddings, dtype=np.float64)

    # Display embedding info for each chunk
    # for i, embedding in enumerate(embeddings):
    #     print(f"\nChunk {i} Embedding:")
    #     print(f"Vector length: {len(embedding)}")
    #     print(f"First 10 values: {embedding[:10]}")

    return embeddings, texts, metadatas, ids

### Save the Embeding into Faiss

In [8]:
# =========================
def faiss_store_step(texts, embeddings, metadatas, ids, domain_name, persist_directory="/content/drive/MyDrive/faiss_vdb"):
    """
    Store embeddings, texts, and metadata into a FAISS index for a specific domain.
    """
    import faiss
    import numpy as np
    import os
    import pickle

    # -----------------------------
    # Create persistent directory if not exists
    # -----------------------------
    os.makedirs(persist_directory, exist_ok=True)

    # -----------------------------
    # Function to get/create domain collection
    # -----------------------------
    def get_domain_collection(domain_name: str, embedding_dim=384):
        index_path = os.path.join(persist_directory, f"{domain_name}_collection.index")
        meta_path = os.path.join(persist_directory, f"{domain_name}_collection_meta.pkl")

        if os.path.exists(index_path):
            index = faiss.read_index(index_path)
            if os.path.exists(meta_path):
                with open(meta_path, "rb") as f:
                    metadata = pickle.load(f)
            else:
                metadata = {}
        else:
            index = faiss.IndexFlatL2(embedding_dim)
            metadata = {}

        return {"index": index, "metadata": metadata, "index_path": index_path, "meta_path": meta_path}

    # -----------------------------
    # Ensure embeddings is 2D NumPy array
    # -----------------------------
    embeddings = np.array(embeddings, dtype=np.float32)
    if embeddings.ndim != 2:
        raise ValueError("Embeddings must be a 2D array of shape (num_vectors, dim)")

    # -----------------------------
    # Get or create domain collection
    # -----------------------------
    collection = get_domain_collection(domain_name, embedding_dim=embeddings.shape[1])
    index = collection["index"]
    metadata = collection["metadata"]

    # -----------------------------
    # Add embeddings and metadata
    # -----------------------------
    index.add(embeddings)
    for i, id_ in enumerate(ids):
        metadata[id_] = {"text": texts[i], **metadatas[i]}

    # =============================
    # 🔥 SAVE LIKE A1 (Current Directory)
    # =============================
    file_name = f"{domain_name}.pkl"   # ← saves in current working directory

    data = {
        "chunks": texts,
        "embeddings": embeddings,
        "index": index
    }

    with open(file_name, "wb") as f:
        pickle.dump(data, f)

    print(f"[SUCCESS] Saved in current directory as: {file_name}")

In [17]:
import faiss
import numpy as np
import os
import pickle

persist_directory = "/content/drive/MyDrive/faiss_vdb"
os.makedirs(persist_directory, exist_ok=True)

def verify_faiss_with_vectors(domain_name: str, embedding_dim=384):
    """
    Load FAISS index and metadata for a domain and display information.
    Also fetches vectors stored in FAISS.
    """
    index_path = os.path.join(persist_directory, f"{domain_name}_collection.index")
    meta_path = os.path.join(persist_directory, f"{domain_name}_collection_meta.pkl")

    # Load FAISS index
    if os.path.exists(index_path):
        index = faiss.read_index(index_path)
    else:
        print(f"No FAISS index found for domain '{domain_name}'.")
        return

    # Load metadata
    if os.path.exists(meta_path):
        with open(meta_path, "rb") as f:
            metadata = pickle.load(f)
    else:
        metadata = {}

    # Display index info
    print(f"FAISS index type: {type(index)}")
    print(f"Number of vectors stored: {index.ntotal}")
    print(f"Number of metadata entries: {len(metadata)}")

    # Access vectors
    if index.ntotal > 0:
        vectors = index.reconstruct_n(0, index.ntotal)  # get all vectors
        print(f"\nFirst vector (dimension {vectors.shape[1]}):\n{vectors[0]}")
    else:
        print("No vectors in index.")

    # Show first 3 metadata entries
    if metadata:
        print("\nSample metadata (first 3 entries):")
        for i, key in enumerate(list(metadata.keys())[:3]):
            print(key, "->", metadata[key])
    else:
        print("No metadata found.")


### Call the function

In [18]:
def full_pipeline(user_input,domain_name):
    # 1️⃣ Cleaning step
    print("\n===== STEP 1: Cleaning =====")
    # user_input = input("Enter document name (with or without .txt): ").strip()
    # domain_name = input("Enter domain name (e.g. staff, admissions, recruitment): ").strip().lower()
    cleaned_docs = clean_text_step(user_input)

    # 2️⃣ Domain input and chunking
    print("\n===== STEP 2: Chunking =====")
    # domain_name = input("Enter domain name (e.g. staff, admissions, recruitment): ").strip().lower()
    chunks = chunking_step(cleaned_docs, domain_name)  # returns list of chunk dicts

    # 3️⃣ Embedding step
    print("\n===== STEP 3: Embedding =====")
    embeddings, texts, metadatas, ids = embedding_step(chunks)  # embeddings is np.array

    # 4️⃣ Store in FAISS
    print("\n===== STEP 4: Storing in FAISS =====")
    faiss_store_step(texts, embeddings, metadatas, ids, domain_name)

    # # 5️⃣ View FAISS vectors / verify
    # print("\n===== STEP 5: Verify FAISS =====")
    # verify_faiss_with_vectors(domain_name)

    print(f"\nPipeline complete for domain '{domain_name}'")
    return cleaned_docs, chunks, embeddings  # optionally return for further use



In [ ]:
cleaned_docs, chunks, embeddings = full_pipeline("bot", "farhan")

# Data visualization Gui and working Logic

In [ ]:


import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import io
import base64
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

def load_dataset(file):
    """Load dataset only from uploaded file"""
    try:
        if file is not None:
            if file.name.endswith('.csv'):
                df = pd.read_csv(file)
            elif file.name.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file)
            else:
                return None, "Unsupported file format. Please upload CSV or Excel file."
        else:
            return None, "Please upload a dataset file."

        return df, "Dataset loaded successfully!"
    except Exception as e:
        return None, f"Error loading dataset: {str(e)}"

def analyze_dataset(df):
    """Perform comprehensive data analysis"""
    if df is None:
        return None, "No dataset loaded."

    analysis_results = {}

    # Basic info
    analysis_results['rows'] = df.shape[0]
    analysis_results['columns'] = df.shape[1]
    analysis_results['column_names'] = df.columns.tolist()

    # Data types
    analysis_results['data_types'] = df.dtypes.astype(str).to_dict()

    # Missing values
    missing_counts = df.isnull().sum()
    missing_percentages = (df.isnull().sum() / len(df)) * 100
    analysis_results['missing_counts'] = missing_counts[missing_counts > 0].to_dict()
    analysis_results['missing_percentages'] = missing_percentages[missing_percentages > 0].to_dict()
    analysis_results['total_missing'] = df.isnull().sum().sum()

    # Duplicates
    analysis_results['duplicates'] = df.duplicated().sum()

    # Unique values per column
    analysis_results['unique_values'] = {col: df[col].nunique() for col in df.columns}

    # For categorical columns - frequency counts
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    analysis_results['categorical_freq'] = {}
    for col in categorical_cols[:3]:
        analysis_results['categorical_freq'][col] = df[col].value_counts().head(5).to_dict()

    # Summary statistics for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        analysis_results['summary_stats'] = df[numeric_cols].describe().to_dict()

        # Range for numeric columns
        analysis_results['ranges'] = {}
        for col in numeric_cols:
            analysis_results['ranges'][col] = {
                'min': df[col].min(),
                'max': df[col].max(),
                'range': df[col].max() - df[col].min()
            }

    return analysis_results, "Analysis completed successfully!"

def format_analysis_report(analysis_results):
    """Format analysis results into a readable report"""
    if not analysis_results or 'rows' not in analysis_results:
        return "No analysis data available."

    report = f"""
**DATASET OVERVIEW**
• Total Rows: {analysis_results['rows']:,}
• Total Columns: {analysis_results['columns']}
• Column Names: {', '.join(analysis_results['column_names'][:10])}{'...' if len(analysis_results['column_names']) > 10 else ''}

**DATA TYPES**
"""

    type_counts = {}
    for col, dtype in analysis_results['data_types'].items():
        type_counts[dtype] = type_counts.get(dtype, 0) + 1

    for dtype, count in type_counts.items():
        report += f"\n    • {dtype}: {count} column(s)"

    report += f"""

⚠️ **MISSING VALUES**
• Total Missing Values: {analysis_results.get('total_missing', 0):,}
"""

    if analysis_results.get('missing_counts'):
        report += "\n    • Columns with missing values:"
        for col, count in analysis_results['missing_counts'].items():
            percentage = analysis_results['missing_percentages'].get(col, 0)
            report += f"\n      - {col}: {count:,} ({percentage:.1f}%)"
    else:
        report += "\n    • No missing values found!"

    report += f"""

🔄 **DUPLICATES**
• Duplicate Rows: {analysis_results.get('duplicates', 0):,}
"""

    if analysis_results.get('summary_stats'):
        report += f"""

**NUMERIC COLUMNS SUMMARY**
"""
        for col, stats in list(analysis_results['summary_stats'].items())[:3]:
            report += f"""
    • {col}:
      - Mean: {stats.get('mean', 0):.2f}
      - Median: {stats.get('50%', 0):.2f}
      - Min: {stats.get('min', 0):.2f}
      - Max: {stats.get('max', 0):.2f}
      - Std: {stats.get('std', 0):.2f}
"""

    if analysis_results.get('categorical_freq'):
        report += f"""

**CATEGORICAL COLUMNS (Top 5 values)**
"""
        for col, freq_dict in analysis_results['categorical_freq'].items():
            report += f"\n    • {col}:"
            for value, count in freq_dict.items():
                report += f"\n      - {value}: {count:,}"

    return report

# DATA CLEANING & NORMALIZATION FUNCTIONS

def clean_and_normalize_data(df, cleaning_options):
    """Perform data cleaning and normalization"""
    if df is None:
        return None, "No dataset to clean."

    cleaning_report = {
        'before': {
            'rows': df.shape[0],
            'columns': df.shape[1],
            'missing_values': df.isnull().sum().sum(),
            'duplicates': df.duplicated().sum()
        }
    }

    df_cleaned = df.copy()


    return report_text

# VISUALIZATION FUNCTIONS WITH CONSTRAINTS

def get_column_constraints(chart_type):
    """Get column constraints based on chart type"""
    constraints = {
        'Bar Chart': {
            'min_columns': 1,
            'max_columns': 2,
            'description': 'X-axis (categorical), Y-axis (numeric)',
            'x_type': 'categorical',
            'y_type': 'numeric',
        },
        'Line Chart': {
            'min_columns': 2,
            'max_columns': 2,
            'description': 'X-axis (numeric), Y-axis (numeric)',
            'x_type': 'numeric',
            'y_type': 'numeric',
        },
        'Scatter Plot': {
            'min_columns': 2,
            'max_columns': 3,
            'description': 'X-axis (numeric), Y-axis (numeric), Color (categorical)',
            'x_type': 'numeric',
            'y_type': 'numeric',
            'color_type': 'categorical',
        },
        'Histogram': {
            'min_columns': 1,
            'max_columns': 1,
            'description': 'Column must be numeric',
            'x_type': 'numeric',
        },
        'Pie Chart': {
            'min_columns': 1,
            'max_columns': 1,
            'description': 'Column should be categorical',
            'x_type': 'categorical',
        },
        'Box Plot': {
            'min_columns': 1,
            'max_columns': 2,
            'description': 'Y-axis (numeric), X-axis (categorical optional)',
            'x_type': 'categorical',
            'y_type': 'numeric',
        },
        'Heatmap': {
            'min_columns': 2,
            'max_columns': 20,
            'description': 'All columns must be numeric for correlation',
            'all_numeric': True,
        },
        'Area Chart': {
            'min_columns': 2,
            'max_columns': 3,
            'description': 'X-axis (numeric), Y-axis (numeric), Group (categorical)',
            'x_type': 'numeric',
            'y_type': 'numeric',
            'color_type': 'categorical',
        }
    }
    return constraints.get(chart_type, {'min_columns': 1, 'max_columns': 2})

def filter_columns_by_type(df, column_type):
    """Filter columns based on required type"""
    if column_type == 'numeric':
        return df.select_dtypes(include=[np.number]).columns.tolist()
    elif column_type == 'categorical':
        return df.select_dtypes(include=['object', 'category']).columns.tolist()
    else:
        return df.columns.tolist()

def generate_visualization(df, x_col, y_col, color_col, chart_type, title):
    """Generate visualization with Plotly and return figure object"""
    if df is None or df.empty:
        return None, "❌ No data available for visualization."

    try:
        fig = None

        if chart_type == 'Bar Chart':
            fig = px.bar(
                df,
                x=x_col if x_col else df.columns[0],
                y=y_col if y_col else (df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else df.columns[1]),
                color=color_col if color_col else None,
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        elif chart_type == 'Line Chart':
            fig = px.line(
                df,
                x=x_col if x_col else df.select_dtypes(include=[np.number]).columns[0],
                y=y_col if y_col else df.select_dtypes(include=[np.number]).columns[1],
                color=color_col if color_col else None,
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        elif chart_type == 'Scatter Plot':
            fig = px.scatter(
                df,
                x=x_col if x_col else df.select_dtypes(include=[np.number]).columns[0],
                y=y_col if y_col else df.select_dtypes(include=[np.number]).columns[1],
                color=color_col if color_col else None,
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        elif chart_type == 'Histogram':
            fig = px.histogram(
                df,
                x=x_col if x_col else df.select_dtypes(include=[np.number]).columns[0],
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        elif chart_type == 'Pie Chart':
            value_counts = df[x_col].value_counts().reset_index() if x_col else None
            if value_counts is not None:
                value_counts.columns = ['Category', 'Count']
                fig = px.pie(
                    value_counts,
                    values='Count',
                    names='Category',
                    title=title or f"{chart_type}",
                    template="plotly_white"
                )

        elif chart_type == 'Box Plot':
            fig = px.box(
                df,
                y=y_col if y_col else df.select_dtypes(include=[np.number]).columns[0],
                x=x_col if x_col else None,
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        elif chart_type == 'Heatmap':
            numeric_df = df.select_dtypes(include=[np.number])
            if not numeric_df.empty and numeric_df.shape[1] >= 2:
                corr_matrix = numeric_df.corr()
                fig = px.imshow(
                    corr_matrix,
                    text_auto=True,
                    aspect="auto",
                    title=title or "Correlation Heatmap",
                    template="plotly_white",
                    color_continuous_scale='RdBu_r'
                )
            else:
                return None, "❌ Heatmap requires at least 2 numeric columns."

        elif chart_type == 'Area Chart':
            fig = px.area(
                df,
                x=x_col if x_col else df.select_dtypes(include=[np.number]).columns[0],
                y=y_col if y_col else df.select_dtypes(include=[np.number]).columns[1],
                color=color_col if color_col else None,
                title=title or f"{chart_type}",
                template="plotly_white"
            )

        if fig:
            fig.update_layout(
                plot_bgcolor='white',
                paper_bgcolor='white',
                font_family="Arial",
                title_x=0.5,
                title_font_size=18,
                margin=dict(l=50, r=50, t=80, b=50),
                height=500,
                width=700
            )
            return fig, f"✅ {chart_type} generated successfully!"
        else:
            return None, "❌ Could not generate visualization. Please check your column selections."

    except Exception as e:
        return None, f"❌ Error generating visualization: {str(e)}"

def update_column_dropdowns(df, chart_type):
    """Update column dropdowns based on selected chart type and dataset"""
    if df is None:
        return (
            gr.Dropdown(choices=[], value=None, interactive=True),
            gr.Dropdown(choices=[], value=None, interactive=True),
            gr.Dropdown(choices=[], value=None, interactive=True),
            "⚠️ Please load a dataset first."
        )

    constraints = get_column_constraints(chart_type)

    numeric_cols = filter_columns_by_type(df, 'numeric')
    categorical_cols = filter_columns_by_type(df, 'categorical')
    all_cols = df.columns.tolist()

    # X-axis choices
    if constraints.get('all_numeric', False):
        x_choices = numeric_cols
    elif constraints.get('x_type') == 'numeric':
        x_choices = numeric_cols if numeric_cols else all_cols
    elif constraints.get('x_type') == 'categorical':
        x_choices = categorical_cols if categorical_cols else all_cols
    else:
        x_choices = all_cols

    # Y-axis choices
    if constraints.get('all_numeric', False):
        y_choices = numeric_cols
    elif constraints.get('y_type') == 'numeric':
        y_choices = numeric_cols if numeric_cols else all_cols
    elif constraints.get('y_type') == 'categorical':
        y_choices = categorical_cols if categorical_cols else all_cols
    else:
        y_choices = all_cols

    # Color choices
    color_choices = categorical_cols if categorical_cols else all_cols

    # Default selections
    x_default = x_choices[0] if x_choices else None
    y_default = y_choices[1] if len(y_choices) > 1 else (y_choices[0] if y_choices else None)
    color_default = None

    message = f" **{chart_type} Requirements:** {constraints.get('description', 'Select appropriate columns')}"

    return (
        gr.Dropdown(choices=x_choices, value=x_default, interactive=True),
        gr.Dropdown(choices=y_choices, value=y_default, interactive=True),
        gr.Dropdown(choices=color_choices, value=color_default, interactive=True),
        message
    )

# =============================================================================
# MAIN UI BUILDING FUNCTION
# =============================================================================

def build_data_vizzz5_tab():
    """Professional data visualization dashboard with step-by-step tabs"""

    with gr.TabItem("Data Visualization & Analysis", id="viz_content"):
        gr.Markdown("## 🔬 Complete Data Analysis & Visualization Dashboard")
        gr.Markdown("---")

        # ==================== STATE VARIABLES ====================
        df_state = gr.State(None)
        cleaned_df_state = gr.State(None)

        # ==================== MAIN TABS CONTAINER ====================
        with gr.Tabs(elem_id="workflow_tabs") as main_tabs:

            # ---------- TAB 1: LOAD DATA ----------
            with gr.Tab("1. 📁 Load Dataset", id="tab_load", visible=True) as tab_load:
                with gr.Group():
                    with gr.Row():
                        with gr.Column(scale=1):
                            data_upload = gr.File(
                                label="Upload Dataset (CSV, Excel)",
                                file_types=[".csv", ".xlsx", ".xls"],
                                type="filepath"
                            )

        analyze_btn.click(
            process_analysis,
            inputs=[df_state],
            outputs=[analysis_report, tab_clean, main_tabs]
        )

        # ----- Toggle Normalization Method Visibility -----
        normalize.change(
            lambda x: gr.update(visible=x),
            inputs=[normalize],
            outputs=[normalization_method]
        )

        # ----- Clean Data (Step 3 -> Step 4) -----
        def process_cleaning(df, remove_dupes, missing_strat, norm, norm_method, encode_cat):
            if df is None:
                return "❌ No dataset loaded.", None, gr.update(visible=False), gr.update(selected="tab_load")

            cleaning_options = {
                'remove_duplicates': remove_dupes,
                'missing_strategy': missing_strat,
                'normalize': norm,
                'normalization_method': norm_method,
                'encode_categorical': encode_cat
            }

            cleaned_df, report = clean_and_normalize_data(df, cleaning_options)
            if cleaned_df is not None:
                report_text = format_cleaning_report(report)
                return report_text, cleaned_df, gr.update(visible=True), gr.update(selected="tab_viz")
            else:
                return f"❌ Cleaning failed: {report}", df, gr.update(visible=False), gr.update(selected="tab_clean")

        clean_btn.click(
            process_cleaning,
            inputs=[
                df_state,
                remove_duplicates,
                missing_strategy,
                normalize,
                normalization_method,
                encode_categorical
            ],
            outputs=[cleaning_report_md, cleaned_df_state, tab_viz, main_tabs]
        )

        # ----- Update Column Dropdowns (when dataset changes) -----
        def update_columns(df, chart_type):
            if df is None:
                return (
                    gr.Dropdown(choices=[], value=None, interactive=True),
                    gr.Dropdown(choices=[], value=None, interactive=True),
                    gr.Dropdown(choices=[], value=None, interactive=True),

                )
            return update_column_dropdowns(df, chart_type)

        # Trigger when cleaned dataset is ready
        cleaned_df_state.change(
            update_columns,
            inputs=[cleaned_df_state, chart_type],
            outputs=[x_axis, y_axis, color_column, constraint_message]
        )

        # Also when original dataset is loaded (before cleaning)
        df_state.change(
            update_columns,
            inputs=[df_state, chart_type],
            outputs=[x_axis, y_axis, color_column, constraint_message]
        )

        # When chart type changes, update dropdown constraints
        chart_type.change(
            update_columns,
            inputs=[cleaned_df_state, chart_type],
            outputs=[x_axis, y_axis, color_column, constraint_message]
        )

        )

    return df_state, cleaned_df_state


# Image Captioning Gui working logic

In [ ]:

def b6():
    import gradio as gr
    from PIL import Image
    import torch

    with gr.TabItem("Image Captioning", id="caption_content"):
        # ===== CUSTOM CSS =====
        gr.HTML("""
        <style>
        /* Parent container */
        #caption_content {
            height: 100vh !important;
            display: flex !important;
            flex-direction: column !important;
            overflow: hidden !important;
            padding: 0 !important;
        }

        /* Main row with 3 columns, no wrap */
        .main-row {
            flex: 1 1 auto !important;
            display: flex !important;
            gap: 20px !important;
            padding: 20px !important;
            flex-wrap: nowrap !important;   /* prevent wrapping */
            min-height: 0 !important;
        }

        /* Left panel - Advanced Settings */
        .left-panel {
            flex: 0 1 25% !important;   /* initial 25%, shrink allowed */
            max-width: 300px !important; /* prevent it from getting too big */
            overflow-y: auto !important;
            min-height: 0 !important;
            display: flex !important;
            flex-direction: column !important;
        }

        /* Middle panel - Image upload */
        .middle-panel {
            flex: 1 1 35% !important;  /* takes remaining space proportionally */
            min-width: 200px !important;
            display: flex !important;
            justify-content: center !important;
            align-items: flex-start !important;
            flex-direction: column !important;
        }

        /* Right panel - output */
        .right-panel {
            flex: 1 1 40% !important;  /* takes remaining space proportionally */
            min-width: 200px !important;
            overflow-y: auto !important;
            display: flex !important;
            flex-direction: column !important;
            min-height: 0 !important;
        }

        /* Generate button - slightly smaller */
        #generate-btn {
            padding: 10px 22px !important;
            font-size: 14px !important;
            border-radius: 32px !important;
        }

        /* Scrollbars for panels */
        .left-panel::-webkit-scrollbar, .right-panel::-webkit-scrollbar {
            width: 6px !important;
        }
        .left-panel::-webkit-scrollbar-thumb, .right-panel::-webkit-scrollbar-thumb {
            background: #3b82f6 !important;
            border-radius: 10px !important;
        }

        /* Make panels shrink proportionally but never wrap */
        @media (max-width: 1000px) {
            .main-row {
                flex-wrap: nowrap !important;
            }
        }

        </style>
        """)

        # ===== Generate Button =====
        with gr.Row(elem_classes="button-container"):
            caption_button = gr.Button(
                "Generate Caption",
                variant="primary",
                size="md",
                elem_id="generate-btn"
            )

        # ===== MAIN 3-COLUMN LAYOUT =====
        with gr.Row(elem_classes="main-row"):
            # LEFT PANEL: SETTINGS
            with gr.Column(elem_classes="left-panel"):
                gr.Markdown("<h3>⚙️ Settings</h3>")
                max_tokens_slider = gr.Slider(50, 300, value=150, step=10, label="Max Description Length")
                beam_slider = gr.Slider(1, 10, value=5, step=1, label="Beam Width")
                temperature_slider = gr.Slider(0.1, 1.5, value=1.0, step=0.1, label="Temperature")
                repetition_penalty_slider = gr.Slider(1.0, 2.0, value=1.2, step=0.1, label="Repetition Penalty")
                resize_checkbox = gr.Checkbox(label="Auto-resize large images", value=True)
                max_size_slider = gr.Slider(256, 768, value=384, step=32, label="Max Image Size (px)")

            # MIDDLE PANEL: IMAGE UPLOAD
            with gr.Column(elem_classes="middle-panel"):
                gr.Markdown("<h3> Upload Image</h3>")
                image_input = gr.Image(type="pil", label="", height=400, show_label=False)

            # RIGHT PANEL: OUTPUT
            with gr.Column(elem_classes="right-panel"):
                gr.Markdown("<h3> Generated Description</h3>")
                caption_output = gr.Textbox(
                    label="",
                    lines=15,
                    placeholder="✨ Your image description will appear here...",
                    show_copy_button=True,
                    show_label=False
                )


        # ===== CONNECT BUTTON =====
        caption_button.click(
            fn=generate_caption_enhanced,
            inputs=[image_input, max_tokens_slider, beam_slider, temperature_slider,
                    repetition_penalty_slider, resize_checkbox, max_size_slider],
            outputs=[caption_output]
        )

        return image_input, caption_output

# Login Page and Run the Model

In [ ]:
# =====================================
custom_css = """
# /////////////////////////////////////////////////////////////////////////////////////////
/* Sidebar and layout styling */

#fixed-search-bar {
    position: sticky;
    top: 0;
    z-index: 1000;
    background: white;
    padding: 10px 0;
    border-bottom: 1px solid #e2e8f0;
}

/* REQUIRED FOR STICKY TO WORK */
.gradio-container {
    height: 100vh;
    overflow-y: auto;
}


#sidebar-toggle {
    display: flex;
    align-items: center;
    justify-content: center;
    width: 40px;
    height: 40px;
    border-radius: 8px;
    background: #f1f5f9;
    border: 1px solid #e2e8f0;
    cursor: pointer;
    transition: all 0.3s ease;
}

#sidebar-toggle:hover {
    background: #e2e8f0;
    transform: scale(1.05);
}

.quick-action-btn {
    width: 100%;
    margin: 5px 0;
    text-align: left;
    padding: 8px 12px;
}

.hamburger-icon {
    font-size: 20px;
    font-weight: bold;
}

/* Scrollable chatbot area */
#chatbot-container {
    height: 600px;
    overflow-y: auto !important;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    padding: 15px;
    background: #f8fafc;
}

/* Custom scrollbar for chatbot */
#chatbot-container::-webkit-scrollbar {
    width: 8px;
}

#chatbot-container::-webkit-scrollbar-track {
    background: #f1f5f9;
    border-radius: 4px;
}

#chatbot-container::-webkit-scrollbar-thumb {
    background: #cbd5e1;
    border-radius: 4px;
}

#chatbot-container::-webkit-scrollbar-thumb:hover {
    background: #94a3b8;
}

/* Chatbot styling */
.chatbot {
    min-height: 600px !important;
    overflow-y: auto !important;
}

/* Sidebar animation */
.sidebar-content {
    transition: all 0.3s ease;
    overflow: hidden;
}

/* Responsive design */
@media (max-width: 768px) {
    .chat-container {
        padding: 0 5px;
    }

    #chatbot-container {
        height: 500px;
    }
}
/* Fix for Dataframe height */
#data-preview-table {
    height: 300px !important;
    overflow-y: auto !important;
}

/* Make dataframe scrollable */
.gr-dataframe {
    max-height: 300px !important;
    overflow-y: auto !important;
}

/* Fix for any other dataframes */
div[class*="dataframe"] {
    max-height: 300px !important;
    overflow-y: auto !important;
}

# //////////////////////////////////////////////////////////////////////////////
# /* Global resets – apply everywhere */
# * {
#     margin: 0;
#     padding: 0;
#     box-sizing: border-box;
#     font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
# }

/* Main Gradio container – white background for main app */
.gradio-container {
    background: white !important;
}

/* ---------- LOGIN PAGE STYLES (scoped inside .login-wrapper) ---------- */
.login-wrapper {
    background: linear-gradient(135deg, #0f1a3d, #1c2e4a, #2a3a5f);
    background-size: 400% 400%;
    animation: gradientBG 15s ease infinite;
    min-height: 100vh;
    display: flex;
    justify-content: center;
    align-items: center;
    padding: 20px;
}

@keyframes gradientBG {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

/* All login-specific classes are now nested under .login-wrapper */
.login-wrapper .container {
    width: 100%;
    max-width: 1200px;
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    box-shadow: 0 15px 35px rgba(0, 0, 0, 0.5);
    border-radius: 20px;
    overflow: hidden;
    min-height: 700px;
    background: rgba(15, 23, 42, 0.8);
}

.login-wrapper .banner {
    flex: 1 1 50% !important;
    background: linear-gradient(rgba(0, 0, 0, 0.7), rgba(0, 0, 0, 0.7)), url('https://images.unsplash.com/photo-1586449480533-9d52c9b713b9?ixlib=rb-4.0.3') center/cover;
    color: white;
    padding: 40px;
    display: flex;
    flex-direction: column;
    justify-content: center;
    position: relative;
}

.login-wrapper .banner-content {
    position: relative;
    z-index: 2;
}

.login-wrapper .banner h1 {
    font-size: 2.8rem;
    margin-bottom: 25px;
    font-weight: 700;
    text-shadow: 0 2px 10px rgba(0,0,0,0.5);
    color: #e2e8f0;
}

.login-wrapper .banner p {
    font-size: 1.2rem;
    line-height: 1.7;
    margin-bottom: 25px;
    max-width: 600px;
    color: #cbd5e1;
}

.login-wrapper .features {
    margin-top: 40px;
}

.login-wrapper .feature {
    display: flex;
    align-items: center;
    margin-bottom: 20px;
    font-size: 1.1rem;
    padding: 12px 15px;
    background: rgba(255,255,255,0.1);
    border-radius: 10px;
    backdrop-filter: blur(5px);
    border-left: 4px solid #3498db;
}

.login-wrapper .feature i {
    color: #3498db;
    margin-right: 15px;
    font-size: 1.4rem;
    min-width: 30px;
    text-align: center;
}

.login-wrapper .form-container {
    width: 450px;
    flex: 0 0 450px !important;
    background: rgba(255, 255, 255, 0.95);
    padding: 50px 40px;
    display: flex;
    flex-direction: column;
    transition: all 0.4s ease;
    position: relative;
    overflow-y: auto;
}

.login-wrapper .form-container::before {
    content: '';
    position: absolute;
    top: -50%;
    left: -50%;
    width: 200%;
    height: 200%;
    background: radial-gradient(circle, rgba(52, 152, 219, 0.1) 0%, rgba(0, 0, 0, 0) 70%);
    transform: rotate(30deg);
    z-index: 0;
}

.login-wrapper .logo {
    margin-bottom: 30px;
    position: relative;
    z-index: 1;
    text-align: center;
}

.login-wrapper .logo i {
    font-size: 4rem;
    background: linear-gradient(to right, #1a2a6c, #3498db);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.login-wrapper .logo h2 {
    margin-top: 15px;
    font-size: 1.8rem;
    color: #2c3e50;
    font-weight: 700;
}

.login-wrapper .form-section {
    position: relative;
    z-index: 1;
    width: 100%;
    transition: transform 0.5s ease, opacity 0.5s ease;
    display: flex;
    flex-direction: column;
}

.login-wrapper .input-group {
    position: relative;
    margin-bottom: 18px;
}

.login-wrapper .input-group i {
    position: absolute;
    left: 15px;
    top: 50%;
    transform: translateY(-50%);
    color: #3498db;
    font-size: 1.2rem;
    z-index: 2;
}

.login-wrapper .form-container input {
    width: 100%;
    padding: 15px 20px 15px 50px;
    border: 2px solid #e1e5ee;
    border-radius: 10px;
    font-size: 1rem;
    transition: all 0.3s;
    background: rgba(255, 255, 255, 0.9);
}

.login-wrapper .form-container input:focus {
    border-color: #3498db;
    box-shadow: 0 0 0 3px rgba(52, 152, 219, 0.2);
    outline: none;
}

.login-wrapper .form-container button {
    width: 100%;
    padding: 15px;
    background: linear-gradient(to right, #1a2a6c, #3498db);
    color: white;
    border: none;
    border-radius: 10px;
    font-size: 1.1rem;
    font-weight: 600;
    cursor: pointer;
    transition: all 0.3s;
    position: relative;
    overflow: hidden;
    z-index: 1;
    margin-top: 10px;
}pper .progress-steps {
    display: flex;
    justify-content: space-between;
    font-size: 0.85rem;
    color: #7f8c8d;
}

.login-wrapper .step.active {
    color: #2c3e50;
    font-weight: 600;
}

.login-wrapper .form-title {
    text-align: center;
    margin-bottom: 30px;
    color: #2c3e50;
    font-size: 1.8rem;
    position: relative;
    z-index: 1;
    font-weight: 600;
}

.login-wrapper .password-strength {
    height: 5px;
    background: #e1e5ee;
    border-radius: 3px;
    margin-top: 10px;
    overflow: hidden;
}

.login-wrapper .strength-meter {
    height: 100%;
    width: 0;
    border-radius: 3px;
    transition: width 0.3s ease, background 0.3s ease;
}

.login-wrapper .strength-weak {
    background: #ff5252;
    width: 33%;
}

.login-wrapper .strength-medium {
    background: #ffb142;
    width: 66%;
}

.login-wrapper .strength-strong {
    background: #2ed573;
    width: 100%;
}

.login-wrapper .strength-text {
    font-size: 0.8rem;
    margin-top: 5px;
    text-align: right;
    color: #7f8c8d;
}

.login-wrapper .security-tips {
    background: #f8f9fa;
    border-radius: 10px;
    padding: 15px;
    margin: 20px 0;
    border-left: 4px solid #3498db;
}

.login-wrapper .security-tips h4 {
    color: #2c3e50;
    margin-bottom: 10px;
    font-size: 1rem;
}

.login-wrapper .security-tips ul {
    padding-left: 20px;
    font-size: 0.9rem;
    color: #555;
}

.login-wrapper .security-tips li {
    margin-bottom: 8px;
}

/* ---------- MAIN APP STYLES (unaffected) ---------- */
# .main-app {
#     width: 100%;
#     max-width: 1400px;
#     margin: 0 auto;
#     background: white;
#     border-radius: 20px;
#     padding: 30px;
#     box-shadow: 0 15px 35px rgba(0,0,0,0.2);
# }

# /* Responsive adjustments for login (scoped) */
# @media (max-width: 1000px) {
#     .login-wrapper .container {
#         flex-direction: column !important;
#         max-width: 600px;
#     }
#     .login-wrapper .banner {
#         flex: 0 0 auto !important;
#         width: 100%;
#     }
#     .login-wrapper .form-container {
#         flex: 0 0 auto !important;
#         width: 100%;

#     }
# }
.login-wrapper .feature span {
    color: white !important;
    font-weight: 500;
}
"""

# =====================================
# LOGIN / ACCOUNT FUNCTIONS (unchanged)
# =====================================
VALID_USERNAME = "admin"
VALID_PASSWORD = "1234"
def check_password_strength(password):
    strength = 0
    if len(password) >= 8: strength += 1
    if re.search(r"[A-Z]", password): strength += 1
    if re.search(r"[0-9]", password): strength += 1
    if re.search(r"[^A-Za-z0-9]", password): strength += 1
    if len(password) == 0: return "Password strength"
    if strength < 2: return "Weak password"
    elif strength < 4: return "Medium strength"
    else: return "Strong password"

    def show_create_section():
        return (
            gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
            "", "", "", "", "",
            '<div class="password-strength"><div class="strength-meter" style="width:0%"></div></div>',
            '<div class="strength-text">Password strength</div>'
        )
    create_link_btn.click(
        fn=show_create_section,
        outputs=[login_section, create_section, success_section,
                 login_user, login_pass, new_user, new_pass, confirm_pass,
                 strength_bar, strength_text]
    )

    def back_to_login_section():
        return (
            gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
            "", "", "", "", ""
        )
    back_to_login_btn.click(
        fn=back_to_login_section,
        outputs=[login_section, create_section, success_section,
                 login_user, login_pass, new_user, new_pass, confirm_pass]
    )

    def login_from_success(su, sp):
        return (
            gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
            gr.update(value=su), gr.update(value=sp), "", "", "", ""
        )
    login_now_btn.click(
        fn=login_from_success,
        inputs=[session_username, session_password],
        outputs=[login_section, create_section, success_section,
                 login_user, login_pass, new_user, new_pass, confirm_pass]
    )

# =====================================
# LAUNCH
# =====================================
if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_3687/3266780132.py:625: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue", secondary_hue="gray")) as demo:
/tmp/ipykernel_3687/3266780132.py:625: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue", secondary_hue="gray")) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Auto-Detect or set allow_custom_value=True.
  warnings.warn(
/tmp/ipykernel_3687/4001552744.py:53: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to Tr

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9f3cd584dd9d428f01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
